# 11 — Spatial Downscaling

Applies the selected model to monthly predictor stacks and creates
downscaled precipitation rasters.

In [ ]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings("ignore")

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").exists() and (candidate / "configs").exists():
            return candidate
    raise FileNotFoundError(
        "Project root was not found. Run this notebook from inside the repository."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
CONFIG_DIR = PROJECT_ROOT / "configs"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR = PROJECT_ROOT / "models"

for folder in [INTERIM_DIR, PROCESSED_DIR, OUTPUT_DIR, MODEL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
import numpy as np
import rasterio
import joblib
import pandas as pd

model_path = MODEL_DIR / "random_forest.joblib"
bundle = joblib.load(model_path)
model = bundle["pipeline"]
features = bundle["features"]

stack_dir = PROCESSED_DIR / "predictor_stack"
prediction_dir = PROCESSED_DIR / "predictions"
prediction_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
for stack_path in sorted(stack_dir.glob("*.tif")):
    with rasterio.open(stack_path) as src:
        cube = src.read().astype("float32")
        rows, cols = src.height, src.width
        descriptions = list(src.descriptions)

        frame = pd.DataFrame({
            descriptions[i] or f"band_{i+1}": cube[i].reshape(-1)
            for i in range(src.count)
        })

        for feature in features:
            if feature not in frame.columns:
                frame[feature] = np.nan

        prediction = model.predict(frame[features])
        prediction = np.maximum(prediction, 0).reshape(rows, cols)

        profile = src.profile.copy()
        profile.update(count=1, dtype="float32", nodata=-9999.0, compress="lzw")

        output_path = prediction_dir / stack_path.name.replace(
            "predictor_stack", "downscaled_precipitation"
        )
        with rasterio.open(output_path, "w", **profile) as dst:
            dst.write(prediction.astype("float32"), 1)

print("Spatial downscaling completed.")